In [53]:
import numpy as np
import pandas as pd
import matplotlib as mb

In [54]:
class Node:
    thresholds=[]
    maxDept=10
    minGain=0.01
    
    def __init__(self, left=None, right=None, value = None, dept=None, threshold = None,feature = None,):
        self.threshold=threshold
        self.left = left
        self.right = right
        self.value=value
        self.feature=feature
        self.dept=dept
        
    def fillThreshold(self,X,Y,method):
        m = X.shape[1]
        Node.thresholds = [None] * m
        for i in range(m):
            bestValue = best_Value(X,Y,i,method)
            Node.thresholds[i]=bestValue

    def createNode(self, left=None, right=None, threshold=None):
        return Node(left=left,right=right,threshold=threshold)

In [55]:
def compute_impurity(Y):
    impurity=.0
    if(len(Y) == 0) : return 0
    if(len(np.unique(Y)) == 1) : return 0

    p1 = np.mean(Y)
    p2=1-p1
    
    impurity = np.log2(p1) * (-p1) - np.log2(p2) * p2
    return impurity

In [56]:
def quintile_threshold(X):
    X_sorted= np.sort(X)
    quintiles = np.linspace(0,100,20)[1:-1]
    values= np.ones(len(quintiles))
    a=0
    for i in quintiles:
        values[a] = np.round(np.percentile(X_sorted,i),2)
        a = a+1
    return values

In [57]:
def unique_threshold(X):
    X_sorted= np.sort(X)
    X_uniqued=np.unique(X_sorted)
    thresholds=(X_uniqued[1:] + X_uniqued[:-1])/2
    return thresholds

In [58]:
def best_Value(X,Y,feature_no,method):
    values = method(X[:,feature_no])
    maxGain= -np.inf
    bestValue= None
    for i in values:
        gain = info_gain(X, Y, feature_no, i)

        if gain > maxGain:
            maxGain = gain
            bestValue = i
            
    return bestValue

In [59]:
def one_hot_encoding(X,Y,method):
    for feature in range(X.shape[1]):
        threshold=best_Value(X,Y,feature,method)
        X[:,feature] = (X[:,feature] > threshold)

In [60]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X = data.data
Y = data.target

In [61]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size = 0.2,random_state=42)

In [66]:
for i in range(5):
    print(Node.thresholds[i])

15.025
18.46
98.43
696.25
0.08946499999999999


In [67]:
def info_gain(X, Y, feature_no, threshold):
    if len(Y) <= 1: return 0
    base_impurity = compute_impurity(Y)
    feature_values = X[:, feature_no]

    mask_left = feature_values <= threshold
    mask_right = ~mask_left

    Y_left = Y[mask_left]
    Y_right = Y[mask_right]
    n=len(Y)
    
    p_left = compute_impurity(Y_left)
    p_right = compute_impurity(Y_right)

    w_left = len(Y_left) / n
    w_right = len(Y_right) / n

    gain = base_impurity - (p_left * w_left + p_right * w_right)

    return gain

In [68]:
def splitData(X,Y,feature_no,threshold):
    mask = X[:, feature_no] <= threshold

    return [X[mask],X[~mask],Y[mask],Y[~mask]]

In [65]:
root = Node(dept=0)
root.fillThreshold(X_train,Y_train,unique_threshold)

In [76]:
def splitTree(node,X,Y,dept):
    if(node is None): node = Node()

    node.dept=dept
    
    if(len(Y) < 10): 
        node.value = np.bincount(Y).argmax()
        return node
        
    if(compute_impurity(Y) == 0):
        node.value=np.bincount(Y).argmax()
        return node
        
    if(node.dept >= Node.maxDept): 
        node.value=np.bincount(Y).argmax()
        return node
    
    bestFeature = None
    bestGain = 0
    bestThreshold= None
    
    for i in range (len(Node.thresholds)):
        gain = info_gain(X, Y, i, Node.thresholds[i])
        if (bestGain < gain):
            bestGain = gain
            bestFeature = i
            bestThreshold = Node.thresholds[i]

    if(bestGain < Node.minGain):
        node.value = np.bincount(Y).argmax()
        return node
        
    node.feature = bestFeature
    node.threshold = bestThreshold
    node.dept=dept

    X_left, X_right, Y_left, Y_right = splitData(X, Y, bestFeature, bestThreshold)
    
    print(    
    "Depth:", node.dept,
    "Samples:", len(Y),
    "Feature:", node.feature,
    "Gain:", "{:.4f}".format(bestGain),
    "Threshold:",node.threshold
    )
    
    node.left = splitTree(Node(), X_left, Y_left, dept+1)
    node.right = splitTree(Node(), X_right, Y_right, dept+1)
    
    return node

In [77]:
root = splitTree(root,X_train,Y_train,dept=0)

Depth: 0 Samples: 455 Feature: 7 Gain: 0.5605 Threshold: 0.05128
Depth: 1 Samples: 282 Feature: 20 Gain: 0.1215 Threshold: 16.795
Depth: 2 Samples: 263 Feature: 13 Gain: 0.0259 Threshold: 31.284999999999997
Depth: 3 Samples: 243 Feature: 1 Gain: 0.0126 Threshold: 18.46
Depth: 4 Samples: 85 Feature: 15 Gain: 0.0243 Threshold: 0.01838
Depth: 5 Samples: 42 Feature: 16 Gain: 0.0264 Threshold: 0.020955
Depth: 6 Samples: 36 Feature: 8 Gain: 0.0359 Threshold: 0.17154999999999998
Depth: 7 Samples: 15 Feature: 4 Gain: 0.0635 Threshold: 0.08946499999999999
Depth: 3 Samples: 20 Feature: 4 Gain: 0.1692 Threshold: 0.08946499999999999
Depth: 4 Samples: 10 Feature: 3 Gain: 0.1935 Threshold: 696.25
Depth: 2 Samples: 19 Feature: 1 Gain: 0.4986 Threshold: 18.46
Depth: 3 Samples: 11 Feature: 15 Gain: 0.4395 Threshold: 0.01838
Depth: 1 Samples: 173 Feature: 22 Gain: 0.2638 Threshold: 114.44999999999999
Depth: 2 Samples: 44 Feature: 21 Gain: 0.4836 Threshold: 24.87
Depth: 3 Samples: 19 Feature: 27 Gain: 0.

In [70]:
def predict(X,node):
    
    root=node
    prediction=[None]* len(X)
    
    for x in range(len(X)):
        node=root
        while(node.value is None):            
            if(node.threshold >= X[x][node.feature]):
                node=node.left
            else:node=node.right
        
        prediction[x] = node.value
        
    return prediction

In [71]:
y_predict = predict(X_test,root)

In [72]:
for i in range(len (y_predict)):
    print(y_predict[i],"    ",Y_test[i])

1      1
0      0
0      0
1      1
1      1
0      0
0      0
0      0
1      1
1      1
1      1
0      0
1      1
0      0
1      1
0      0
1      1
1      1
1      1
0      0
1      0
1      1
0      0
1      1
1      1
1      1
1      1
1      1
1      1
0      0
1      1
1      1
1      1
1      1
1      1
1      1
0      0
1      1
0      0
1      1
1      1
0      0
1      1
1      1
1      1
1      1
1      1
1      1
1      1
1      1
0      0
0      0
1      1
1      1
1      1
1      1
1      1
0      0
0      0
1      1
1      1
0      0
0      0
1      1
1      1
1      1
0      0
0      0
1      1
1      1
0      0
0      0
1      1
0      0
1      1
1      1
1      1
1      0
1      1
1      1
0      0
1      1
1      0
0      0
0      0
0      0
0      0
0      0
1      1
1      1
1      1
1      1
1      1
1      1
1      1
1      1
0      0
0      0
1      1
0      0
0      0
1      1
0      0
0      0
1      1
1      1
1      1
0      0
1      1
1      1
0      0
1

In [73]:
def calculateAcc(y_pre,y):
    mask= y_pre == y
    np.sum(mask)
    print(len(y) - np.sum(mask))

In [74]:
calculateAcc(y_predict,Y_test)

3


In [75]:
len(Y_test)

114